# NB1 — Open Bandit Dataset: data audit & temporal split

**Goal:** confirm the bandit semantics, reproduce the empirical facts we measured in
August 2026, and write the chronological train/eval artifacts every later notebook uses.

Semantics: one row = one impression. Arms = (item_id, position), reward = `click`,
context = 80 user-item affinity columns. Logging policy on this shard: **bts**
(Bernoulli Thompson Sampling).

In [1]:
import json
import numpy as np
import pandas as pd

from src.obd_io import load_shard, temporal_split, item_universe, affinity_cols
from src import config

In [2]:
df = load_shard(n_rows=config.N_ROWS)
ctr = df["click"].mean()
print("rows:", len(df))
print(f"empirical logged CTR = {ctr:.4f}")
print("items:", df["item_id"].nunique(), "| positions:", sorted(df["position"].unique()))

rows: 200000
empirical logged CTR = 0.0046
items: 80 | positions: [1, 2, 3]


## Audit 1 — affinity sparsity (~95% all-zero rows expected)
Affinity = past-click counts between this user and each of the 80 items.

In [3]:
aff = affinity_cols(df)
A = df[aff].to_numpy()
all_zero = (A == 0).all(axis=1).mean()
print(f"all-zero affinity rows: {all_zero:.1%}  (expect ~95%)")
print("nonzero affinity values:", np.unique(A[A > 0])[:10], "| max:", A.max())

all-zero affinity rows: 95.4%  (expect ~95%)
nonzero affinity values: [1. 2. 3. 4.] | max: 4.0


## Audit 2 — propensity is NOT static per (item, position)
If the same (item, position) shows many distinct propensity values, the logging
policy's probabilities move over time (stochastic exploration), which is exactly
why the dataset logs propensity per row instead of giving a lookup table.

In [4]:
g = df.groupby(["item_id", "position"])["propensity_score"].nunique()
print("combos with >1 distinct propensity:", int((g > 1).sum()), "of", len(g))
print("distinct propensity values overall:", df["propensity_score"].nunique())

combos with >1 distinct propensity: 237 of 239
distinct propensity values overall: 5544


## Audit 3 — position bias: CTR by slot (1 = left, most visible)

In [5]:
print(df.groupby("position")["click"].agg(["mean", "count"]).round(5))

             mean  count
position                
1         0.00452  66888
2         0.00447  66401
3         0.00481  66711


## Temporal 80/20 split -> artifacts
Train (earlier) / eval (later) with a shared arm universe so arm indices stay
identical across NB2-NB4. NB2 fits on train and does OPE on eval only.

In [6]:
train, ev = temporal_split(df)
ids = item_universe(train, ev)
train.to_parquet(config.DATA / "obd_train.parquet")
ev.to_parquet(config.DATA / "obd_eval.parquet")
(config.ARTIFACTS / "arm_universe.json").write_text(json.dumps([int(i) for i in ids]))

summary = {
    "rows": int(len(df)),
    "ctr": float(ctr),
    "all_zero_affinity_share": float(all_zero),
    "n_items": int(df["item_id"].nunique()),
    "n_arm_universe": len(ids),
    "train_rows": int(len(train)),
    "eval_rows": int(len(ev)),
    "train_ctr": float(train["click"].mean()),
    "eval_ctr": float(ev["click"].mean()),
}
(config.ARTIFACTS / "nb1_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "rows": 200000,
  "ctr": 0.0046,
  "all_zero_affinity_share": 0.95389,
  "n_items": 80,
  "n_arm_universe": 80,
  "train_rows": 160000,
  "eval_rows": 40000,
  "train_ctr": 0.00458125,
  "eval_ctr": 0.004675
}


## Summary
- Shard confirmed: 80 items x 3 slots, tiny CTR, affinity ~95% zeros (vals 1-4).
- Propensity is per-impression and moves over time -> must use logged pscore for OPE.
- Artifacts written: `data/obd_train.parquet`, `data/obd_eval.parquet`,
  `artifacts/arm_universe.json`, `artifacts/nb1_summary.json`.